# RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/yz/83q64df112j55g1h0p6vkxph0000gn/T/ipykernel_85657/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: deep_learning_intro.pdf
  ✓ Loaded 3 pages

Processing: nlp_intro.pdf
  ✓ Loaded 3 pages

Processing: speech_processing_intro.pdf
  ✓ Loaded 3 pages

Total documents loaded: 9


In [3]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks=split_documents(all_pdf_documents)
chunks

Split 9 documents into 26 chunks

Example chunk:
Content: RAG Learning Notes
Page 1
 DEEP LEARNING
 Neural networks, optimisation, architectures, and evaluation
1. What is deep learning?
Deep learning is a subfield of machine learning that uses neural networ...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T12:34:15+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T12:34:15+00:00', 'subject': '(unspecified)', 'title': 'Deep Learning', 'trapped': '/False', 'source': '../data/pdf_files/deep_learning_intro.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'deep_learning_intro.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T12:34:15+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T12:34:15+00:00', 'subject': '(unspecified)', 'title': 'Deep Learning', 'trapped': '/False', 'source': '../data/pdf_files/deep_learning_intro.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'deep_learning_intro.pdf', 'file_type': 'pdf'}, page_content='RAG Learning Notes\nPage 1\n DEEP LEARNING\n Neural networks, optimisation, architectures, and evaluation\n1. What is deep learning?\nDeep learning is a subfield of machine learning that uses neural networks with multiple layers to learn representations from\ndata. Rather than relying entirely on manually designed features, a network can learn useful intermediate features during\ntraining.\nApplications include image recognition, speech recognition, language modelling, recommendation, forecasting, and\ngenerative 

# Enbedding and VectorStoreDB

In [4]:
import numpy as np 
from sentence_transformers import SentenceTransformer 
import chromadb 
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

/Users/aviksarkar/Desktop/Python/rag_application/rag_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1101 20:18:01.507000 85657 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Embedding Manager

In [5]:
class EmbeddingManager:
    """Handles Document Embedding Generation Using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the Embedding Manager
        Args:
            model_name: Hugging Face model name for sentence embedding.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading the model {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Successfully Loaded. Embedding Dimension : {self.model.get_sentence_embedding_dimension()}")
        except ValueError as e:
            print(f"Error Loading model {self.model_name} : {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """ Generate embeddings for a list of texts
            Args:
                texts: List of text strings to embed

            Returns:
                numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    # def get_embedding_dimension(self) -> int:
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #     return self.model.get_sentence_embedding_dimension()



## Initialize the Embedding Manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading the model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11375.06it/s]


Model Successfully Loaded. Embedding Dimension : 384


/var/folders/yz/83q64df112j55g1h0p6vkxph0000gn/T/ipykernel_85657/3953471801.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Successfully Loaded. Embedding Dimension : {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [6]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 416


In [7]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T12:34:15+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T12:34:15+00:00', 'subject': '(unspecified)', 'title': 'Deep Learning', 'trapped': '/False', 'source': '../data/pdf_files/deep_learning_intro.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'deep_learning_intro.pdf', 'file_type': 'pdf'}, page_content='RAG Learning Notes\nPage 1\n DEEP LEARNING\n Neural networks, optimisation, architectures, and evaluation\n1. What is deep learning?\nDeep learning is a subfield of machine learning that uses neural networks with multiple layers to learn representations from\ndata. Rather than relying entirely on manually designed features, a network can learn useful intermediate features during\ntraining.\nApplications include image recognition, speech recognition, language modelling, recommendation, forecasting, and\ngenerative 

In [8]:
### Convert text to embeddings

texts = [doc.page_content for doc in chunks]
texts

### Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### Store in the Vector Database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 26 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.18it/s]

Generated embeddings with shape: (26, 384)
Adding 26 documents to vector store...
Successfully added 26 documents to vector store
Total documents in collection: 442


# Retriever Pipeline From VectorStore

In [9]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [10]:
rag_retriever.retrieve("Explain in Detail NLP")

Retrieving documents for query: 'Explain in Detail NLP'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.23it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d673326b_92',
  'content': 'Core concepts, text representations, models, and evaluation\n1. What is NLP?',
  'metadata': {'source_file': 'nlp_intro.pdf',
   'page_label': '1',
   'author': '(anonymous)',
   'page': 0,
   'keywords': '',
   'doc_index': 92,
   'total_pages': 3,
   'trapped': '/False',
   'content_length': 75,
   'source': '../data/pdf_files/nlp_intro.pdf',
   'creationdate': '2026-10-31T12:34:14+00:00',
   'subject': '(unspecified)',
   'title': 'NLP Introduction',
   'creator': '(unspecified)',
   'producer': 'ReportLab PDF Library - (opensource)',
   'moddate': '2026-10-31T12:34:14+00:00',
   'file_type': 'pdf'},
  'similarity_score': 0.3443637490272522,
  'distance': 0.6556362509727478,
  'rank': 1},
 {'id': 'doc_4924ddb5_185',
  'content': 'NLP requires appropriate',
  'metadata': {'doc_index': 185,
   'source_file': 'nlp_intro.pdf',
   'keywords': '',
   'total_pages': 3,
   'content_length': 24,
   'title': 'NLP Introduction',
   'creationdate': '2026

In [11]:
rag_retriever.retrieve("What are CNNs and transformers?")

Retrieving documents for query: 'What are CNNs and transformers?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 121.51it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_374e9129_53',
  'content': 'Image and audio generation\n8. CNNs and transformers',
  'metadata': {'source_file': 'deep_learning_intro.pdf',
   'page_label': '2',
   'author': '(anonymous)',
   'creator': '(unspecified)',
   'source': '../data/pdf_files/deep_learning_intro.pdf',
   'keywords': '',
   'title': 'Deep Learning',
   'page': 1,
   'doc_index': 53,
   'creationdate': '2026-10-31T12:34:15+00:00',
   'content_length': 51,
   'file_type': 'pdf',
   'trapped': '/False',
   'producer': 'ReportLab PDF Library - (opensource)',
   'total_pages': 3,
   'moddate': '2026-10-31T12:34:15+00:00',
   'subject': '(unspecified)'},
  'similarity_score': 0.31031495332717896,
  'distance': 0.689685046672821,
  'rank': 1},
 {'id': 'doc_77d23351_8',
  'content': 'Diffusion\nLearns to reverse a gradual noising process\nImage and audio generation\n8. CNNs and transformers\nA convolutional neural network applies learnable filters across an input. Early layers can detect simple patterns, 

# RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.getenv("GROQ_API_KEY")
# print(os.getenv("GROQ_API_KEY"))

In [19]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [20]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [21]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [22]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("What is NLP?")

Retrieving documents for query: 'What is NLP?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.34it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d673326b_92',
  'content': 'Core concepts, text representations, models, and evaluation\n1. What is NLP?',
  'metadata': {'total_pages': 3,
   'page': 0,
   'author': '(anonymous)',
   'trapped': '/False',
   'page_label': '1',
   'content_length': 75,
   'file_type': 'pdf',
   'doc_index': 92,
   'title': 'NLP Introduction',
   'source_file': 'nlp_intro.pdf',
   'producer': 'ReportLab PDF Library - (opensource)',
   'source': '../data/pdf_files/nlp_intro.pdf',
   'creationdate': '2026-10-31T12:34:14+00:00',
   'subject': '(unspecified)',
   'creator': '(unspecified)',
   'moddate': '2026-10-31T12:34:14+00:00',
   'keywords': ''},
  'similarity_score': 0.522004246711731,
  'distance': 0.47799575328826904,
  'rank': 1},
 {'id': 'doc_4924ddb5_185',
  'content': 'NLP requires appropriate',
  'metadata': {'keywords': '',
   'trapped': '/False',
   'doc_index': 185,
   'moddate': '2026-10-31T12:34:14+00:00',
   'creator': '(unspecified)',
   'source_file': 'nlp_intro.pdf',
   '

# Integration Vectordb Context pipeline With LLM output

In [27]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [28]:
answer=rag_simple("What is NLP?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is NLP?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.21it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Natural Language Processing (NLP) is an AI field that enables computers to analyze, understand, process, and generate human language, integrating linguistics, computer science, statistics, and machine learning.


# Enhanced RAG Pipeline Features

In [29]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.63it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [31]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is NLP?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is NLP?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.74it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Core concepts, text representations, models, and evaluation
1. What is NLP?

NLP requires appropriate

RAG Learning Notes
Page 1
 NATURAL LANGUAGE PROCESSING (NLP)
 Core concepts, text representations, models, and evaluation
1. What is NLP?
Natural La

nguage Processing is a field of artificial intelligence concerned with enabling computers to analyse, understand,
process, and generate human language. It combines linguistics, computer science, statistics, and machine learning. NLP
systems commonly work with text, transcripts, or other language representations.

Question: what is NLP?

Answer:

Final Answer: Natural Language Processing (NLP) is an AI field that enables computers to analyze, understand, process, and generate human language, integrating linguistics, computer science, statistics, and machine learning.

Citations:
[1] nlp_intro.pdf (page 0)
[2] nlp_intro.pdf (page 2)
[3] nlp_intro.pdf (page 0)
Summary: Natural Language Processing (NLP) is a branch of AI that lets computers analyze, understand, process, and generate human language. It blends linguistics, computer science, statistics, and machine learning to achieve this.
History: {'question': 'what is NLP?', 'answer': 'Natural Language Processing (NLP) is an AI field that 